### COTI (LLM-only Stage)

In [1]:
from utils import *
from train import *
from AzureUtils import *
from AzureInference import *
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
import numpy as np
import torch
import json
import glob
import os
import random

#### Step 1 Initial Topic Discovery

In [2]:
folder_path = "covid_data"
docx_files = [f for f in os.listdir(folder_path) if f.endswith(".docx")]

random.seed(42)
selected_files = random.sample(docx_files, 2)
selected_paths = [os.path.join(folder_path, f) for f in selected_files]


In [3]:
system_message = """
### Background ###
The COVID-19 pandemic severely tested global health systems, leading to rapid operational adaptations like surge capacity expansion and widespread telemedicine adoption, while exposing critical vulnerabilities in supply chains. Healthcare workforces faced immense mental health burdens and ethical dilemmas regarding resource allocation and duty to treat. 
Policy responses varied, with governance structures and political leadership significantly influencing effectiveness, often highlighting pre-existing weaknesses in public health infrastructure and exacerbating health inequities. The crisis underscored the urgent need for sustained investment in preparedness, data-driven decision-making, and universal health coverage to build more resilient and equitable systems for future challenges.            

You are a qualitative research expert tasked with identifying topics of interviews, which were conducted with health workers, policymakers, key informants, and patients between Oct-Dec 2020 to examining the health system response to COVID-19 in Sierra Leone.
The research aims to explore how the pandemic affected service delivery, health workers, patient access to services, leadership, and governance. Additionally, the research examines to what extent the legacy of the 2013–2016 Ebola outbreak influenced the COVID-19 response and public perception.
Your task is to extract key clues (limit to 200 words) diectly from original dialogues supporting each given identified topic.
            
Your task:
- Identify important topics for the given interview (there may be more than one).
- For each identified topic, provide clues and reasoning to explain the connection.
            
Clues must:
- Be direct quotes from the dialogue (no summarization or interpretation).
- Be brief but contextually complete.
- Highlight key phrases, contextual information, emotional tones, or symptoms related to the topic.

Reasoning must:
- Links the clues directly to the topic.
- Explains the logical connection between the clues and topic.
- Avoids adding external context or information not present in the clues.
"""

user_message = """
Your task is to identify important topics for the given interview.
Each topic should be concise, meaningful, and specific. Avoid combining distinct ideas or using vague terms.
Step 1 Extract CLUES.
Step 2 Generate REASONING.
Step 3 Identify TOPICS: Based on the dialogue, clues, and reasoning, identify all applicable topics.

### Output Format ###
For EACH identified topic, provide the following EXACTLY:
Identify topic: [Insert topic here]
Clues (max 200 words): [Insert clues here]
Reasoning (max 150 words): [Insert reasoning here]

Dialogue: {dialogue}
"""

In [4]:
file_path_1 = selected_paths[0]
dialogue_1 = extract_text_from_docx(file_path_1)
user_message_1 = user_message.replace("{dialogue}", dialogue_1)

In [5]:
model_name = "Qwen/QwQ-32B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map=torch.device("cuda:4")
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

In [ ]:
response_1 = qwen_response(
    model=model,
    tokenizer=tokenizer,
    system_prompt=system_message,
    user_prompt=user_message_1,
    print_prompt=True  
)


====== Final Prompt Sent to Model ======

<|im_start|>system

### Background ###
The COVID-19 pandemic severely tested global health systems, leading to rapid operational adaptations like surge capacity expansion and widespread telemedicine adoption, while exposing critical vulnerabilities in supply chains. Healthcare workforces faced immense mental health burdens and ethical dilemmas regarding resource allocation and duty to treat. 
Policy responses varied, with governance structures and political leadership significantly influencing effectiveness, often highlighting pre-existing weaknesses in public health infrastructure and exacerbating health inequities. The crisis underscored the urgent need for sustained investment in preparedness, data-driven decision-making, and universal health coverage to build more resilient and equitable systems for future challenges.            

You are a qualitative research expert tasked with identifying topics of interviews, which were conducted with 

In [8]:
response_1 = strip_thinking_content(response_1)
output_path = "/home/qxu4/topic_identification/covid_project/output/initial_topics_1.txt"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(response_1)

In [9]:
file_path_2 = selected_paths[1]
dialogue_2 = extract_text_from_docx(file_path_2)
user_message_2 = user_message.replace("{dialogue}", dialogue_2)

In [10]:
response_2 = qwen_response(
    model=model,
    tokenizer=tokenizer,
    system_prompt=system_message,
    user_prompt=user_message_2,
    print_prompt=False  
)

response_2 = strip_thinking_content(response_2)
output_path = "/home/qxu4/topic_identification/covid_project/output/initial_topics_2.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(response_2)

#### Step 2 Preprocessing -- Finetune instructions

In [12]:
docx_file_paths = selected_paths

topics_list = [
    "1. Fear and Stigma Affecting Patient Attendance, 2. Adaptations in Service Delivery During Pandemic, 3. Legacy of Ebola on COVID-19 Response, 4. Health Workers' Mental Health and Safety Concerns, 5. Impact on Non-COVID Services and Supply Chain Challenges, 6. Ebola vs. COVID-19 Response and Outcomes, 7. Recommendations for Health System Strengthening",
    "1. Impact of the pandemic on service delivery (referral practices, operational changes), 2. Patient access and behavior changes due to fear and curfew. 3. Supply chain and financial challenges (drug shortages, cost increases), 4. Legacy of the Ebola outbreak influencing response strategies, 5. Mental health and lack of support for healthcare workers, 6. Staff shortages and payment issues affecting service continuity, 7. PPE availability and reliance on past outbreak supplies"
]

initial_clue_instruction = "List clues (i.e. key phrases, contextual information, semantic and emotional tones, temporal information) in the following interviews that support each given identified topic.\n\n"

initial_reasoning_instruction = "Based on the given clues, generate the reasoning process that supports the identified topics.\n\n"

json_output_dir = "/home/qxu4/topic_identification/covid_project/output/training_results"

num_iterations = 4

final_results = iterative_workflow(
        model, tokenizer, docx_file_paths, topics_list, initial_clue_instruction, initial_reasoning_instruction, json_output_dir, num_iterations
    )

Running iteration 0/4...

Iteration 0 Feedback:


**Common Issues:**  
**Clue Generation:**  
1. **Missing Context:** Many clues lack specificity about *why* certain measures were taken, *how* they impacted services, or *when* changes occurred. For example, "we have returned to our normal routine" (Topic 1) contradicts earlier clues about ongoing curfews and referrals but provides no context for this shift.  
2. **Inconsistent or Conflicting Clues:** Some topics contain conflicting statements (e.g., Topic 2: "Patients were coming" vs. "Some were afraid"), but no contextualization of why these contradictions exist.  
3. **Overgeneralization:** Clues sometimes omit critical details (e.g., "drugs were not available in the pharmacy" lacks specifics about which drugs or reasons for shortages).  
4. **Irrelevant Details:** Redundant mentions of generic actions (e.g., "put people in charge to ensure compliance") without linking them to broader systemic issues.  

**Reasoning Generation:**  
1

#### Step 3.1 Inference -- Topic Identification

In [2]:
optimized_clue_prompt = """
List **direct quotes** from the dialogue as clues for each topic, ensuring:  
1. **Quantitative Specificity & Systemic Impact**: Include measurable data (percentages, costs, timelines) and explicitly tie examples to systemic challenges. Avoid vague terms like "some" or "many."  
   - Example: "Antibiotic shortages forced **30%** of patients to self-medicate due to **6-month stockouts**" (instead of "patients faced shortages").  
   - Specify impacts: "Drug price hikes (e.g., Novalgin at **Le 2000**) delayed **50%** of critical treatments, worsening maternal health outcomes."  

2. **Contextual Linkages**: For each clue, embed explicit references to broader systemic factors (e.g., policy failures, resource gaps) and their implications.  
   - Example: "Night curfews forced referrals to **under-resourced hospitals**, exacerbating inequities in rural areas."  

3. **Timeframe & Comparison Clarity**: Clarify whether changes were pandemic-specific or pre-existing. Use temporal details (e.g., "since 2019," "post-Ebola").  
   - Example: "Salaries were unpaid for **two years**, worsening pre-existing staff retention issues before the pandemic."  

4. **Precision & Relevance**: Focus on unique aspects of each topic. Exclude redundant or tangential details unless tied to systemic issues.  
   - Example: "Ebola-era PPE reliance persisted, highlighting **preparedness gaps** for respiratory diseases" (only in relevant topics).  

Clues must remain **brief direct quotes**, avoiding summaries or interpretations. Ensure quotes are contextually complete and resolve contradictions with explicit explanations.
"""

optimized_reasoning_prompt = """
Analyze clues to build reasoning for each topic, ensuring:  
1. **Explicit Linkages & Impact**: For each clue, state *how* it supports the topic and *why* it matters.  
   - Example: "Clue 5’s antibiotic shortages directly worsened maternal health outcomes by delaying treatments (Clue 7)."  

2. **Trade-off Analysis**: Explain unintended consequences of adaptations.  
   - Example: "Curfews reduced non-urgent visits (Clue 2) but worsened access for **night labor patients** (Clue 4), creating cascading inequities."  

3. **Address Gaps & Implications**: Identify missing information in clues and infer systemic implications.  
   - Example: "Clue 3 mentions price hikes but omits alternative suppliers, suggesting reliance on **monopolistic markets** that limit affordability."  

4. **Comparative Analysis**: For Ebola vs. COVID topics, contrast strategies and outcomes explicitly.  
   - Example: "Unlike Ebola’s strict isolation (Clue 9), global vaccine coordination during COVID accelerated access (Clue 12), though distribution gaps persisted."  

5. **Cause-Effect Chains**: Use clues to build logical chains linking systemic factors to outcomes.  
   - Example: "Unpaid salaries (Clue 6) caused staff attrition (Clue 10), directly delaying triage operations (Clue 11)."  

Structure reasoning to resolve contradictions, prioritize core themes, and avoid external assumptions. Each point must strictly reference provided clues and their explicit connections to systemic resilience.
"""

In [3]:
test_dialogue_file_paths = [
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_A.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_B.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_C.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_D.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_E.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_F.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_G.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_H.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_I.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_J.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_K.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_L.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_M.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_O.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_P.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_R.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_S.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_T.docx',
    '/home/qxu4/topic_identification/covid_project/covid_data/Transcript_U.docx'
]

In [ ]:
# set API
init_client(
    azure_endpoint = "your_azure_endpoint", 
    api_key = "your_api_key",  
    api_version = "your_api_version"        
)

In [5]:
def process_few_shot_setting(dialogue_paths, clue_optimized, reasoning_optimized):
    output_file_path = "/home/qxu4/topic_identification/covid_project/output/testing_results"
    
    aggregate_results_dir = "/home/qxu4/topic_identification/covid_project/output/testing_results/aggregate_results"

    # Process multiple dialogues
    results_summary = process_multiple_dialogues_n(
        dialogue_paths,
        clue_optimized,
        reasoning_optimized,
        output_file_path,
        n=3
    )

    aggregate_topics_summary = process_aggregate_topics(results_summary, aggregate_results_dir)
    
    return results_summary,aggregate_topics_summary

In [6]:
def process_few_shot_results(result_type="aggregate"):
    base_folder = "/home/qxu4/topic_identification/covid_project/output/testing_results/aggregate_results"
   
    json_file_paths = glob.glob(os.path.join(base_folder, "*.json"))

    if not json_file_paths:
        print(f"No JSON files found for {result_type} results.")
        return None, None

    # Step 1: Remove unnecessary symbols
    process_multiple_json_files(json_file_paths)

    # Step 2: Merge all JSON files into one
    output_json_file = os.path.join(base_folder, f"all_patients_{result_type}.json")
    merge_json_files(json_file_paths, output_json_file)

    # Step 3: Convert JSON to a readable text format
    output_txt_file = os.path.join(base_folder, f"all_patients_{result_type}.txt")
    save_entire_json_as_expanded_text(output_json_file, output_txt_file)

    return output_json_file, output_txt_file

In [8]:
results_iteration, aggregate_iteration = process_few_shot_setting(
    dialogue_paths=test_dialogue_file_paths,  
    clue_optimized=optimized_clue_prompt,
    reasoning_optimized=optimized_reasoning_prompt
)

aggregate_json, aggregate_txt = process_few_shot_results(result_type="aggregate")

#### Step 3.2 Inference -- Codebook

In [ ]:
# set API
init_client(
    azure_endpoint = "your_azure_endpoint", 
    api_key = "your_api_key",  
    api_version = "your_api_version"        
)

In [4]:
messages = codebook("/home/qxu4/topic_identification/covid_project/output/testing_results/aggregate_results/all_patients_aggregate.json")
response = get_completion(messages)
cluster_text = response.choices[0].message.content
print(cluster_text) 

Here's a synthesized thematic codebook based on the provided data from the various transcripts regarding the impact of COVID-19 on healthcare systems in Sierra Leone:

```json
[
  {
    "code_name": "Impact of COVID-19 on Service Delivery",
    "description": "This code captures the significant disruptions in healthcare service delivery due to the COVID-19 pandemic, including reduced patient inflow, suspension of elective surgeries, and ongoing challenges in maintaining standard operations.",
    "original_topics": [
      "Impact of COVID-19 on Healthcare Access",
      "Impact of COVID-19 on Health Services",
      "Impact of COVID-19 on Patient Access to Services",
      "Service Delivery Challenges During COVID-19"
    ],
    "representative_clues": [
      "the influx of patients at the initial stage of the outbreak, drop",
      "patients were not coming, they feared that when them come and they tell them they have COVID",
      "the theater never did Elective surgeries but just 

In [5]:
with open("/home/qxu4/topic_identification/covid_project/output/testing_results/codebook.json", "w") as f:
    json.dump(cluster_text, f, indent=2)